# **Libra**
> Brady Spears, Los Alamos National Laboratory
## Serializing & Deserializing Schemas from a Python Dictionary

---

### About this Notebook
This notebook provides insight into the simple instantiation of `Libra's` _Schema_ object, and gives examples of how to 'load' and 'dump' abstract SQLAlchemy _Table_ instances (referred to interchangably as _Models_) to and from regular Python dictionaries. Some `SQLAlchemy` boilerplate code is shown to exemplify how a user can interact with `SQLAlchemy` declartive objects, performing CRUD (create-remove-update-delete) operations in the OOP environment that are then reflected in the SQL environment. 

---

### Defining a Schema definition Dictionary
A "schema definition" is a serialized instance of a relational database schema, and in this case, is contained within a regular Python dictionary. Contained within the 
dictionary are 'columns' and 'models', which describe the structure of both `SQLAlchemy` _Column_ objects and abstract `SQLAlchemy` _Table_ objects. Key-value pairs contained in sub-dictionaries that describe each are ultimately passed to the '__init__()' methods of each object during their dynamic de-serialization. Using a subset of the _NNSA KB Core_, a "Knowledge Base" relational database schema developed by the National Nuclear Security Administration for purposes of seismic study, the serialized schema takes the form:

In [5]:
schema_dict = {
    'NNSA KB Core Subset' : {
        'description' : 'Subset of NNSA KB Core tables for example purposes',
        'columns' : {
            'lddate' : {
                'type' : 'DateTime()',
                'nullable' : False,
                'default' : {'$ref' : 'datetime.now'},
                'onupdate' : {'$ref' : 'datetime.now'},
                'info' : {
                    'format' : '%Y-%m-%d %H:%M:%S',
                    'width' : 19
                }
            },
            'net' : {
                'type' : 'String(length = 8)',
                'default' : '-',
                'nullable' : False,
                'info' : {
                    'format' : '8.8s',
                    'width' : 8,
                    'regex' : r"^[A-Z0-9][A-Z0-9:\-\+_]*$"
                }
            },
            'sta' : {
                'type' : 'String(length = 6)',
                'default' : '-',
                'nullable' : False,
                'info' : {
                    'format' : '6.6s',
                    'width' : 6,
                    'regex' : r"^[A-Z0-9\-][A-Z0-9x\-\*+]+$"
                }
            },
            'time' : {
                'type' : 'Float(precision = 53)',
                'default' : -9999999999.999,
                'nullable' : False,
                'info' : {
                    'format' : '17.5f',
                    'width' : 17,
                    'gt' : -9999999999.999
                }
            },
            'endtime' : {
                'type' : 'Float(precision = 53)',
                'default' : 9999999999.999,
                'nullable' : True,
                'info' : {
                    'format' : '17.5f',
                    'width' : 17,
                    'gt' : -9999999999.999,
                    'lt' : 9999999999.999
                }
            }
        },
        'models' : {
            'affiliation' : {
                'columns' : ['net', 'sta', 'time', 'endtime', 'lddate'],
                'constraints' : [
                    {'pk' : {'columns' : ['net', 'sta', 'time']}}
                ]
            }
        }
    }
}

Each 'column' entry must contain a 'type' key-value pair that maps to an SQLAlchemy _type_ object. Any extraneous information can be passed to a column's 'info' dictionary. In `Libra's` case, those values are exclusively used by various "mix-in" packages designed to augment model behavior in OOP space. 'Model' entries in the dictionary contain string references to each column that makes up that model as well as constraints. `Libra` supports Primary Key Constraints ('pk'), Unique Constraints ('uq'), Check Constraints ('ck'), and Indexes ('ix'). Foreign Key constraints are not currently supported due to restrictions on how `SQLAlchemy` internally resolves reference names when constructing _ForeignKeyConstraint_ objects. Future editions of `Libra` may implement support for Foreign keys, but do not in version 1.0.0.

Now that we have a schema definition, we can use `Libra's` _Schema_ object to load in the table definitions:

In [6]:
from libra import Schema

kbcore_subset = Schema('NNSA KB Core Subset').load(schema_dict)

print(f'Schema \'{kbcore_subset.name}\' contains :')
print(f'  Models : {list(kbcore_subset._registry.models.keys())}')
columns = list(kbcore_subset._registry.columns.keys())
print(f'  Columns : {columns}')

Schema 'NNSA KB Core Subset' contains :
  Models : ['affiliation']
  Columns : ['lddate', 'net', 'sta', 'time', 'endtime']


Through the _Schema_ object, concrete (not abstract) `SQLAlchemy` _Table_ objects can be created by simply creating a new class tha inherits from the Model and has a '__tablename__' attribute attached to it. The '__tablename__' attribute is the name of the table in the SQL environment and what `SQLAlchemy` uses to reference the table when rendering Data Definition Language (DDL). This DDL is emitted anytime a transaction with the database backend occurs, whether that is creating tables, dropping tables, inserting data, or updating data.

In [7]:
class Affiliation(kbcore_subset.affiliation):
    __tablename__ = 'my_affiliation_table'

# All methods and components of the Affiliation object
print(Affiliation.__table__.__dict__)

{'dispatch': <sqlalchemy.event.base.DDLEventsDispatch object at 0x7fa0570ca0f0>, 'name': 'my_affiliation_table', '_columns': <sqlalchemy.sql.base.DedupeColumnCollection object at 0x7fa0d4e1f160>, 'primary_key': PrimaryKeyConstraint(Column('net', String(length=8), table=<my_affiliation_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault('-')), Column('sta', String(length=6), table=<my_affiliation_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault('-')), Column('time', Float(precision=53), table=<my_affiliation_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault(-9999999999.999))), 'foreign_keys': set(), 'fullname': 'my_affiliation_table', 'metadata': MetaData(), 'schema': None, '_sentinel_column': None, 'indexes': set(), 'constraints': {PrimaryKeyConstraint(Column('net', String(length=8), table=<my_affiliation_table>, primary_key=True, nullable=False, default=ScalarElementColumnDefault('-')), Column('sta', Strin

Now that we have a concrete table, we can use standard `SQLAlchemy` declarative syntax to interact with it. Documentation on how to interact with declarative objects in `SQLAlchemy` is available at https://www.sqlalchemy.org/

In [17]:
import os
from datetime import datetime

import sqlalchemy
from sqlalchemy.orm import sessionmaker

local_db = 'data/examples.db' # Path to local sqlite3 database

# Remove local database for clean slate
if os.path.exists(local_db):
    os.remove(local_db)

engine = sqlalchemy.create_engine(f'sqlite:///{local_db}')
LocalSession = sessionmaker(bind = engine)

with LocalSession() as session:

    # Create all tables associated with kbcore_subset Schema object
    kbcore_subset.base.metadata.create_all(engine)

    # Create a list of objects to add to the database
    entries = [
        Affiliation('USGS', 'ANMO', 144547200, None, None),
        Affiliation(net = 'TX', sta = 'BRDY', time = 1487116800.),
        Affiliation(net = 'ISC', sta = 'WDY', time = -549331200., endtime = 20563200., lddate = datetime(2018, 10, 29))
    ]

    [session.add(entry) for entry in entries]
    session.commit() # Commit the additions to the database

    # Now, let's query all of that back
    query_results = session.query(Affiliation).all()

    # Also, show an example of filtering
    filtered_query_results = session.query(Affiliation).filter(Affiliation.sta == 'WDY').one()

print('All query results:')
for row in query_results:
    print(f'  {row}')

print('Filtered query results:')
print(f'  {filtered_query_results}')

All query results:
  Affiliation(net='USGS', sta='ANMO', time=144547200.0, endtime=9999999999.999, lddate=2026-03-03 15:13:28.488271)
  Affiliation(net='TX', sta='BRDY', time=1487116800.0, endtime=9999999999.999, lddate=2026-03-03 15:13:28.488329)
  Affiliation(net='ISC', sta='WDY', time=-549331200.0, endtime=20563200.0, lddate=2018-10-29 00:00:00)
Filtered query results:
  Affiliation(net='ISC', sta='WDY', time=-549331200.0, endtime=20563200.0, lddate=2018-10-29 00:00:00)


Notice how above, in defining the _Affiliation_ object, `Libra` supports both positional and keyword initialization patterns. For values that are _None_ or simply not included, `Libra's` custom metaclass defaults that column's value to the pre-defined 'default' parameter. 

`Libra` supports the dumping of _Schema_ objects, too. To export the loaded schema definition back into a Python dictionary:

In [18]:
dumped_dictionary = kbcore_subset.dump()

print(dumped_dictionary)

{'NNSA KB Core Subset': {'description': 'Subset of NNSA KB Core tables for example purposes', 'columns': {'lddate': {'type': 'DateTime()', 'nullable': False, 'default': {'$ref': 'datetime.now'}, 'onupdate': {'$ref': 'datetime.now'}, 'info': {'format': '%Y-%m-%d %H:%M:%S', 'width': 19}}, 'net': {'type': 'String(length = 8)', 'default': '-', 'nullable': False, 'info': {'format': '8.8s', 'width': 8, 'regex': '^[A-Z0-9][A-Z0-9:\\-\\+_]*$'}}, 'sta': {'type': 'String(length = 6)', 'default': '-', 'nullable': False, 'info': {'format': '6.6s', 'width': 6, 'regex': '^[A-Z0-9\\-][A-Z0-9x\\-\\*+]+$'}}, 'time': {'type': 'Float(precision = 53)', 'default': -9999999999.999, 'nullable': False, 'info': {'format': '17.5f', 'width': 17, 'gt': -9999999999.999}}, 'endtime': {'type': 'Float(precision = 53)', 'default': 9999999999.999, 'nullable': True, 'info': {'format': '17.5f', 'width': 17, 'gt': -9999999999.999, 'lt': 9999999999.999}}}, 'models': {'affiliation': {'columns': ['net', 'sta', 'time', 'endti